In [1]:
import pandas as pd
import numpy as np
import re 
from IPython.display import display, HTML

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, hamming_loss, jaccard_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)

path = r"‪D:\ML Data\Spesis\anti_total.csv".strip('\u202a')
data= pd.read_csv(path, index_col=False)

In [3]:
roomno_mapping = {'A': '1', 'C': '2', 'D': '3', 'E': '4', 'H': '5', 'K': '6'}
data['ROOMNO'] = data['ROOMNO'].map(roomno_mapping)          

data['SEX'] = data['SEX'].map({'M': 1, 'F': 0})

yn_cols = [
    'ISSEPSIS0', 'FEVER', 'DM', 'CARDIOVASCULAR', 'RESPIRATORY', 'CNS', 'CANCER', 'LIVER', 'KIDNEY', 'AUTOIMMUNE',
    'Hb', 'Ht' , 'Neutrophil Seg.','Absolute Neutrophil count', 'Na','K','Creatinine' , 'Leukocyte level', 'Nitrite level', 'Bacteria level', 
    'Microscopic RBC level','Microscopic WBC level', 'Influenza Virus A level'

]

for col in yn_cols:
    data[col] = data[col].map({'Y': 1, 'N': 0})

In [4]:
start_index = data.columns.get_loc('Acyclovir')
end_index = data.columns.get_loc('tenofovir/emtricitabine/rilpivirine')

In [5]:
abx_cols = data.columns[start_index:end_index+1]
col_sum = data[abx_cols].sum()

In [6]:
final_cols = col_sum[col_sum >= 500].index.tolist()
base_cols = [c for c in data.columns if c not in abx_cols]
data_filter = data[base_cols + final_cols]

In [7]:
feature_cols = list(set(data_filter.columns) - set(abx_cols))
X = data_filter[feature_cols]
y = data_filter[final_cols]

In [8]:
X.columns

Index(['SEX', 'KIDNEY', 'CANCER', 'Microscopic WBC level', 'FEVER',
       'Microscopic RBC level', 'DM', 'VITALSIGNSGCS', 'INFECTIONSITE9',
       'BE(ecf)', 'CHECKITEM30SCORE', 'GPT', 'Creatinine', 'VITALSIGNSBT',
       'INFECTIONSITE5', 'CNS', 'ROOMNO', 'PH', 'AUTIBIOTIC_GROUP',
       'ACCOUNTNO', 'PCO2', 'PLT', 'Na', 'CHECKITEM32SCORE',
       'CHECKITEM27SCORE', 'Lymphocyte', 'PT', 'HCO3', 'MAP', 'Hb',
       'RESPIRATORY', 'CHECKITEM28SCORE', 'HST', 'INFECTIONSITE3', 'STAYTIME',
       'VITALSIGNSRR', 'Absolute Neutrophil count', 'INFECTIONSITE2', 'INR',
       'Influenza Virus A level', 'Bacteria level', 'CHECKITEM28A',
       'Leukocyte level', 'O2 SAT', 'OTHERINFECTIONSITE_flag', 'eGFR-MDRD',
       'CRP', 'VITALSIGNSPR', 'Neutrophil Seg.', 'K', 'AGE', 'CHECKITEM27',
       'APTT', 'INJURELEVEL', 'VITALSIGNSSPO2', 'ISSEPSIS0', 'Nitrite level',
       'Ht', 'AUTOIMMUNE', 'INFECTIONSITE1', 'T.Bilirubin', 'VITALSIGNSDBP',
       'INFECTIONSITE4', 'LIVER', 'CARDIOVASCULAR', 'CHE

In [9]:
X = X.drop(columns=['ACCOUNTNO','ROOMNO','AUTIBIOTIC_GROUP'])

In [10]:
X_cols = ['VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP', 'WBC', 
         'Lymphocyte', 'CRP', 'HCO3', 'PT', 'GPT', 'PCO2',
         'CHECKITEM27SCORE', 'CHECKITEM28SCORE', 'CHECKITEM29SCORE', 'CHECKITEM30SCORE', 'CHECKITEM31SCORE', 'CHECKITEM32SCORE',
         # 'Leukocyte level', 'Microscopic RBC level', 'Microscopic WBC level', 'Nitrite level', 'Influenza Virus A level',
         'INFECTIONSITE1', 'INFECTIONSITE2', 'INFECTIONSITE3', 'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9']

X = X[X_cols]

In [11]:
X.columns

Index(['VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2',
       'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP', 'WBC', 'Lymphocyte', 'CRP',
       'HCO3', 'PT', 'GPT', 'PCO2', 'CHECKITEM27SCORE', 'CHECKITEM28SCORE',
       'CHECKITEM29SCORE', 'CHECKITEM30SCORE', 'CHECKITEM31SCORE',
       'CHECKITEM32SCORE', 'INFECTIONSITE1', 'INFECTIONSITE2',
       'INFECTIONSITE3', 'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9'],
      dtype='object')

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 123)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((22374, 26), (22374, 20), (5594, 26), (5594, 20))

In [13]:
X_train.isnull().sum(), X_test.isnull().sum()

(VITALSIGNSBT          245
 VITALSIGNSPR          904
 VITALSIGNSRR          863
 VITALSIGNSSPO2       5627
 VITALSIGNSDBP        2525
 VITALSIGNSGCS         333
 MAP                  2525
 WBC                 15179
 Lymphocyte          15214
 CRP                 16658
 HCO3                20780
 PT                  20197
 GPT                 15542
 PCO2                20704
 CHECKITEM27SCORE    18407
 CHECKITEM28SCORE    18126
 CHECKITEM29SCORE    17507
 CHECKITEM30SCORE    17670
 CHECKITEM31SCORE    21306
 CHECKITEM32SCORE    18465
 INFECTIONSITE1          0
 INFECTIONSITE2          0
 INFECTIONSITE3          0
 INFECTIONSITE4          0
 INFECTIONSITE5          0
 INFECTIONSITE9          0
 dtype: int64,
 VITALSIGNSBT          67
 VITALSIGNSPR         216
 VITALSIGNSRR         228
 VITALSIGNSSPO2      1404
 VITALSIGNSDBP        608
 VITALSIGNSGCS         75
 MAP                  608
 WBC                 3771
 Lymphocyte          3783
 CRP                 4128
 HCO3                51

In [14]:
X_train.dtypes, X_test.dtypes

(VITALSIGNSBT        float64
 VITALSIGNSPR        float64
 VITALSIGNSRR        float64
 VITALSIGNSSPO2      float64
 VITALSIGNSDBP       float64
 VITALSIGNSGCS       float64
 MAP                 float64
 WBC                 float64
 Lymphocyte          float64
 CRP                 float64
 HCO3                float64
 PT                  float64
 GPT                 float64
 PCO2                float64
 CHECKITEM27SCORE    float64
 CHECKITEM28SCORE    float64
 CHECKITEM29SCORE    float64
 CHECKITEM30SCORE    float64
 CHECKITEM31SCORE    float64
 CHECKITEM32SCORE    float64
 INFECTIONSITE1        int64
 INFECTIONSITE2        int64
 INFECTIONSITE3        int64
 INFECTIONSITE4        int64
 INFECTIONSITE5        int64
 INFECTIONSITE9        int64
 dtype: object,
 VITALSIGNSBT        float64
 VITALSIGNSPR        float64
 VITALSIGNSRR        float64
 VITALSIGNSSPO2      float64
 VITALSIGNSDBP       float64
 VITALSIGNSGCS       float64
 MAP                 float64
 WBC                 float6

In [15]:
y_train.sum().sort_values(ascending=False)

Amoxicillin/Clavulanic acid    4915.0
Flomoxef                       4427.0
Cefixime                       3063.0
Cefazolin                      2192.0
Ciprofloxacin                  2158.0
Cefuroxime                     2078.0
Piperacillin/Tazobactam        2056.0
Azithromycin                   2046.0
Cefoperazone/sulbactam         1236.0
Metronidazole                  1222.0
Levofloxacin                   1145.0
Baloxavir marboxil              932.0
Peramivir                       927.0
Cefadroxil                      895.0
Clindamycin                     796.0
Oseltamivir                     682.0
Ceftriaxone                     663.0
Gentamicin                      653.0
Cephalexin                      515.0
Doxycycline                     422.0
dtype: float64

In [16]:
# 轉數值
num_cols = ['VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP'
           , 'WBC', 'Lymphocyte', 'CRP', 'HCO3', 'PT', 'GPT', 'PCO2']
for col in num_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

In [17]:
# vital sign impute
vital_cols = ['VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP'
           , 'WBC', 'Lymphocyte', 'CRP', 'HCO3', 'PT', 'GPT', 'PCO2']

for col in vital_cols:
    # X_train[col + "_missing"] = X_train[col].isna().astype(int) # missing indicator
    median = X_train[col].median()
    X_train[col] = X_train[col].fillna(median)
    
    # X_test[col + "_missing"] = X_test[col].isna().astype(int) # missing indicator
    X_test[col] = X_test[col].fillna(median)

In [18]:
X_train.columns

Index(['VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2',
       'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP', 'WBC', 'Lymphocyte', 'CRP',
       'HCO3', 'PT', 'GPT', 'PCO2', 'CHECKITEM27SCORE', 'CHECKITEM28SCORE',
       'CHECKITEM29SCORE', 'CHECKITEM30SCORE', 'CHECKITEM31SCORE',
       'CHECKITEM32SCORE', 'INFECTIONSITE1', 'INFECTIONSITE2',
       'INFECTIONSITE3', 'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9'],
      dtype='object')

In [19]:
# drop_cols = ['CHECKITEM28A', 'CHECKITEM27']
# X_train = X_train.drop(columns=drop_cols)
# X_test = X_test.drop(columns=drop_cols)

In [20]:
# fill score
score_cols = ['CHECKITEM27SCORE', 'CHECKITEM28SCORE', 'CHECKITEM29SCORE', 'CHECKITEM30SCORE', 'CHECKITEM31SCORE', 'CHECKITEM32SCORE']
             # 'Leukocyte level', 'Microscopic RBC level', 'Microscopic WBC level', 'Nitrite level', 'Influenza Virus A level']

for col in score_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_train[col] = X_train[col].fillna(-1)

    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
    X_test[col] = X_test[col].fillna(-1)

In [21]:
X_train.isna().sum()                      

VITALSIGNSBT        0
VITALSIGNSPR        0
VITALSIGNSRR        0
VITALSIGNSSPO2      0
VITALSIGNSDBP       0
VITALSIGNSGCS       0
MAP                 0
WBC                 0
Lymphocyte          0
CRP                 0
HCO3                0
PT                  0
GPT                 0
PCO2                0
CHECKITEM27SCORE    0
CHECKITEM28SCORE    0
CHECKITEM29SCORE    0
CHECKITEM30SCORE    0
CHECKITEM31SCORE    0
CHECKITEM32SCORE    0
INFECTIONSITE1      0
INFECTIONSITE2      0
INFECTIONSITE3      0
INFECTIONSITE4      0
INFECTIONSITE5      0
INFECTIONSITE9      0
dtype: int64

In [22]:
y_train.sum(axis=1).mean() # 每人平均用1.65個抗生素

np.float64(1.4759542325914008)

In [23]:
scaled_cols = ['VITALSIGNSBT', 'VITALSIGNSPR', 'VITALSIGNSRR', 'VITALSIGNSSPO2', 'VITALSIGNSDBP', 'VITALSIGNSGCS', 'MAP'
           , 'WBC', 'Lymphocyte', 'CRP', 'HCO3', 'PT', 'GPT', 'PCO2']

scaler = StandardScaler()

X_train[scaled_cols] = scaler.fit_transform(X_train[scaled_cols])
X_test[scaled_cols] = scaler.fit_transform(X_test[scaled_cols])

In [24]:
X_train

,VITALSIGNSBT,VITALSIGNSPR,VITALSIGNSRR,VITALSIGNSSPO2,VITALSIGNSDBP,VITALSIGNSGCS,MAP,WBC,Lymphocyte,CRP,HCO3,PT,GPT,PCO2,CHECKITEM27SCORE,CHECKITEM28SCORE,CHECKITEM29SCORE,CHECKITEM30SCORE,CHECKITEM31SCORE,CHECKITEM32SCORE,INFECTIONSITE1,INFECTIONSITE2,INFECTIONSITE3,INFECTIONSITE4,INFECTIONSITE5,INFECTIONSITE9
26479,0.480317,1.872028,-0.307599,-0.689175,0.759437,0.28521,0.502939,-0.016786,0.060501,-0.193409,-0.022320,-0.061599,-0.019958,-0.059606,2.0,0.0,0.0,0.0,-1.0,1.0,1,0,0,0,0,0
11531,0.011745,0.031668,-1.224904,0.683409,-0.380095,0.28521,-0.507659,-0.007852,-0.133558,-0.193409,-0.022320,-0.061599,-0.078017,-0.059606,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0,0
15854,-1.393971,-1.853579,0.303937,0.683409,0.835405,0.28521,0.692426,-0.014313,4.537704,-0.193409,-0.022320,-0.994761,-0.089628,-0.059606,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0,0
26104,-0.550542,-1.090503,-0.307599,0.957925,0.683468,0.28521,0.439777,-0.009114,-0.410784,-0.184877,-0.022320,-0.061599,0.374840,-0.059606,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0,1,0,0,0,0
22286,-0.363113,-1.359824,0.303937,0.134375,-2.051409,0.28521,-2.213044,-0.008571,-0.632565,4.572923,-0.022320,-0.061599,-0.066405,-0.059606,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15377,0.199173,0.525423,-0.307599,0.134375,-0.911877,0.28521,-0.886634,-0.004508,-1.699886,0.030408,-0.022320,-0.061599,0.014877,-0.059606,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0,0
21602,-0.175684,0.570310,-0.307599,0.957925,-1.975440,0.28521,-2.086719,-0.010048,-0.937514,-0.873526,-0.094278,0.761779,-0.170910,-1.886994,2.0,0.0,0.0,0.0,2.0,0.0,1,1,0,0,0,0
17730,-1.112828,-2.661542,0.303937,0.957925,0.683468,0.28521,1.955674,-0.014918,3.054544,-0.193409,-0.022320,-0.061599,-0.066405,-0.059606,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0,0
15725,-0.269399,1.512934,-0.307599,0.683409,-2.127378,0.28521,-2.339369,-0.002363,0.282282,0.479619,1.560761,-0.665410,-0.078017,-1.331838,0.0,0.0,0.0,0.0,-1.0,1.0,1,1,1,0,0,0


In [25]:
# RandomForestClassifier?

In [26]:
base_model = RandomForestClassifier(
                                    n_estimators=500, 
                                    class_weight='balanced',
                                    min_samples_leaf=5,
                                    min_samples_split=10,
                                    max_depth=15,
                                    n_jobs=-1,
                                    random_state=123)
multi_model = MultiOutputClassifier(base_model)

multi_model.fit(X_train, y_train)

MultiOutputClassifier(estimator=RandomForestClassifier(class_weight='balanced',
                                                       max_depth=15,
                                                       min_samples_leaf=5,
                                                       min_samples_split=10,
                                                       n_estimators=500,
                                                       n_jobs=-1,
                                                       random_state=123))

In [27]:
y_train.columns

Index(['Amoxicillin/Clavulanic acid', 'Azithromycin', 'Baloxavir marboxil',
       'Cefadroxil', 'Cefazolin', 'Cefixime', 'Cefoperazone/sulbactam',
       'Ceftriaxone', 'Cefuroxime', 'Cephalexin', 'Ciprofloxacin',
       'Clindamycin', 'Doxycycline', 'Flomoxef', 'Gentamicin', 'Levofloxacin',
       'Metronidazole', 'Oseltamivir', 'Peramivir', 'Piperacillin/Tazobactam'],
      dtype='object')

In [28]:
drug_name = 'Piperacillin/Tazobactam'
drug_index = list(y_train.columns).index(drug_name)
model = multi_model.estimators_[drug_index]
importances = model.feature_importances_

In [29]:
importance_df = pd.DataFrame({'Features': X_train.columns, 'Importance':importances})
print(importance_df.sort_values(by='Importance', ascending=False))

            Features  Importance
20    INFECTIONSITE1    0.276139
21    INFECTIONSITE2    0.162180
5      VITALSIGNSGCS    0.062552
22    INFECTIONSITE3    0.053788
9                CRP    0.051508
4      VITALSIGNSDBP    0.034470
6                MAP    0.034132
1       VITALSIGNSPR    0.033312
0       VITALSIGNSBT    0.033160
3     VITALSIGNSSPO2    0.032856
7                WBC    0.024120
19  CHECKITEM32SCORE    0.021503
16  CHECKITEM29SCORE    0.020369
8         Lymphocyte    0.019299
11                PT    0.018899
2       VITALSIGNSRR    0.018599
14  CHECKITEM27SCORE    0.017920
12               GPT    0.015797
24    INFECTIONSITE5    0.014395
13              PCO2    0.010917
15  CHECKITEM28SCORE    0.010871
10              HCO3    0.010414
18  CHECKITEM31SCORE    0.008412
17  CHECKITEM30SCORE    0.007546
25    INFECTIONSITE9    0.005628
23    INFECTIONSITE4    0.001214


In [66]:
# site_cols

site_cols = [
'INFECTIONSITE1',
'INFECTIONSITE2',
'INFECTIONSITE3',
'INFECTIONSITE4',
'INFECTIONSITE5',
'INFECTIONSITE9'
]

drug_cols = y_train.columns
# case number
site_counts = X_train[site_cols].sum()
print(site_counts)

INFECTIONSITE1    4728
INFECTIONSITE2    3570
INFECTIONSITE3    2453
INFECTIONSITE4     213
INFECTIONSITE5    1188
INFECTIONSITE9     475
dtype: int64


In [30]:
#計算「每個 infection site → 抗生素分布 entropy」

from scipy.stats import entropy
import pandas as pd
import numpy as np

site_entropy = []

for site in site_cols:
    
    mask = X_train[site] == 1
    
    # 該 site 的病人
    y_site = y_train[mask]
    
    if len(y_site) == 0:
        continue
    
    # 每種抗生素開立機率
    probs = y_site.mean()
    
    probs = probs[probs > 0]   # 避免 log(0)
    
    ent = entropy(probs, base=2)
    
    site_entropy.append({
        "site": site,
        "entropy": ent,
        "cases": len(y_site)
    })

pd.DataFrame(site_entropy).sort_values("entropy")

,site,entropy,cases
1,INFECTIONSITE2,3.318240,3570
4,INFECTIONSITE5,3.455503,1188
2,INFECTIONSITE3,3.585618,2453
0,INFECTIONSITE1,3.669259,4728
3,INFECTIONSITE4,3.681263,213
5,INFECTIONSITE9,3.734018,475


In [31]:
# 單一抗生素 vs infection site entropy

# 也就是：

# 某抗生素 → infection site 是否固定

drug_entropy = []

for drug in y_train.columns:
    
    p = y_train[drug].mean()
    
    probs = [p, 1-p]
    
    ent = entropy(probs, base=2)
    
    drug_entropy.append({
        "drug": drug,
        "entropy": ent,
        "prescription_rate": p
    })

pd.DataFrame(drug_entropy).sort_values("entropy")

,drug,entropy,prescription_rate
12,Doxycycline,0.134998,0.018861
9,Cephalexin,0.158065,0.023018
14,Gentamicin,0.190291,0.029186
7,Ceftriaxone,0.192546,0.029633
17,Oseltamivir,0.196802,0.030482
11,Clindamycin,0.221632,0.035577
3,Cefadroxil,0.242300,0.040002
18,Peramivir,0.248820,0.041432
2,Baloxavir marboxil,0.249832,0.041655
15,Levofloxacin,0.291369,0.051175


In [64]:
# heatmap_data

import pandas as pd

heatmap_data = pd.DataFrame(index=site_cols, columns=drug_cols)

for site in site_cols:
    
    mask = X_train[site] == 1
    
    if mask.sum() == 0:
        continue
    
    probs = y_train[mask].mean()
    
    heatmap_data.loc[site] = probs

heatmap_data = heatmap_data.astype(float)

# 平均使用率
drug_order = heatmap_data.mean().sort_values(ascending=False).index

heatmap_data = heatmap_data[drug_order]
print(heatmap_data)


# heatmap_plot
# import seaborn as sns
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12,6))

# sns.heatmap(
#     heatmap_data,
#     cmap="Reds",
#     linewidths=0.5
# )

# plt.title("P(Antibiotic | Infection Site)")
# plt.xlabel("Antibiotic")
# plt.ylabel("Infection Site")

# plt.show()

                Flomoxef  Piperacillin/Tazobactam  \
INFECTIONSITE1  0.412437                 0.307741   
INFECTIONSITE2  0.667227                 0.257703   
INFECTIONSITE3  0.492458                 0.274358   
INFECTIONSITE4  0.206573                 0.314554   
INFECTIONSITE5  0.214646                 0.212963   
INFECTIONSITE9  0.425263                 0.290526   

                Amoxicillin/Clavulanic acid  Cefixime  Ceftriaxone  \
INFECTIONSITE1                     0.196066  0.157572     0.075719   
INFECTIONSITE2                     0.056583  0.253782     0.079272   
INFECTIONSITE3                     0.070118  0.221769     0.062780   
INFECTIONSITE4                     0.150235  0.145540     0.563380   
INFECTIONSITE5                     0.617003  0.083333     0.056397   
INFECTIONSITE9                     0.237895  0.208421     0.113684   

                Ciprofloxacin  Levofloxacin  Cefazolin  Metronidazole  \
INFECTIONSITE1       0.086294      0.174069   0.064298       0.0

In [32]:
y_pred_train = multi_model.predict(X_train)
print(f1_score(y_train, y_pred_train, average='micro'))
print(f1_score(y_train, y_pred_train, average='macro'))

0.4991858531033879
0.4893660779529245


In [33]:
y_pred = multi_model.predict(X_test)
print(y_pred[:5])

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1.]
 [1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0.]]


In [34]:
y_prob_list = multi_model.predict_proba(X_test)

# y_prob = np.array([prob[:,1] for prob in y_prob_list]).T
# print(y_prob[0])

In [35]:
# proba_matrix = np.column_stack([p[:,1] for p in y_prob_list])
# y_pred = (proba_matrix > 0.2).astype(int)

In [36]:
print(f1_score(y_test, y_pred, average='micro'))
print(f1_score(y_test, y_pred, average='macro'))

0.36972677595628417
0.3059606049302334


In [37]:
hamming_loss(y_test, y_pred)

0.10309259921344298

In [38]:
jaccard_score(y_test, y_pred, average='samples', zero_division=0)

np.float64(0.22614514701125357)

In [39]:
# top k hits rate

def hit_rate_at_k(y_true, proba, k=3):
    
    if isinstance(proba, list):
       proba = np.column_stack([p[:, 1] for p in proba])
    
    topk = np.argsort(proba, axis=1)[:, -k:]
    hits = 0
    for i in range(len(y_true)):
        actual = np.where(y_true[i]==1)[0]
        if len(set(actual)& set(topk[i].tolist())):
            hits += 1
    return hits /  len(y_true)

In [40]:
hit3 = hit_rate_at_k(y_test.values, y_prob_list, k=3)
print('Hit@3: ', hit3)

Hit@3:  0.6220951018948874


In [41]:
# recall@3

def recall_at_k(y_true, proba, k=3):
    if isinstance(proba, list):
        proba = np.column_stack([p[:, 1] for p in proba])

    topk = np.argsort(proba, axis=1)[:, -k:]
    hits = 0
    recalls = []
    for i in range(len(y_true)):
        actual = np.where(y_true[i]==1)[0]
        if len(actual) == 0:
            continue
        recall = len(set(actual) & set(topk[i])) / len(actual)
        recalls.append(recall)
    return np.mean(recalls)

In [42]:
recall_3 = recall_at_k(y_test.values, y_prob_list, k=3)
print('Recall@3: ', recall_3)

Recall@3:  0.5330068154483496


In [43]:
# MAP@3

def map_at_k(y_true, proba, k=3):
    if isinstance(proba, list):
        proba = np.column_stack([p[:, 1] for p in proba])

    topk = np.argsort(proba, axis=1)[:, -k:]
    hits = 0
    APs = []
    for i in range(len(y_true)):
        actual = np.where(y_true[i]==1)[0]
        if len(actual) == 0:
            continue
        score = 0
        hits = 0

        for j in range(k):
            if topk[i][j] in actual:
                hits += 1
                score += hits / (j + 1)
        APs.append(score / min(len(actual), k))
    return np.mean(APs)

In [44]:
map_3 = map_at_k(y_test.values, y_prob_list, k=3)
print('MAP@3: ', map_3)

MAP@3:  0.32002390603887326


In [45]:
target_names = y_train.columns

for i, col in enumerate(target_names):
    print(f'-- {col} --')
    print(classification_report(y_test.iloc[:, i], y_pred[:, i]))

-- Amoxicillin/Clavulanic acid --
              precision    recall  f1-score   support

         0.0       0.82      0.97      0.89      4389
         1.0       0.69      0.21      0.32      1205

    accuracy                           0.81      5594
   macro avg       0.75      0.59      0.60      5594
weighted avg       0.79      0.81      0.77      5594

-- Azithromycin --
              precision    recall  f1-score   support

         0.0       0.94      0.90      0.92      5109
         1.0       0.24      0.34      0.28       485

    accuracy                           0.85      5594
   macro avg       0.59      0.62      0.60      5594
weighted avg       0.88      0.85      0.86      5594

-- Baloxavir marboxil --
              precision    recall  f1-score   support

         0.0       0.98      0.87      0.93      5359
         1.0       0.20      0.69      0.30       235

    accuracy                           0.87      5594
   macro avg       0.59      0.78      0.62      5

In [46]:
multi_model_importance =  multi_model.estimators_[0].feature_importances_
importance_df = pd.DataFrame({'Features': X_train.columns, 'Importance':multi_model_importance})
print(importance_df.sort_values(by='Importance', ascending=False))

            Features  Importance
21    INFECTIONSITE2    0.155337
24    INFECTIONSITE5    0.147036
22    INFECTIONSITE3    0.067625
1       VITALSIGNSPR    0.062253
4      VITALSIGNSDBP    0.056428
6                MAP    0.055826
0       VITALSIGNSBT    0.053726
8         Lymphocyte    0.045015
7                WBC    0.040836
11                PT    0.037416
9                CRP    0.036698
5      VITALSIGNSGCS    0.027741
3     VITALSIGNSSPO2    0.027513
12               GPT    0.026655
20    INFECTIONSITE1    0.026592
2       VITALSIGNSRR    0.022174
17  CHECKITEM30SCORE    0.019727
16  CHECKITEM29SCORE    0.019632
19  CHECKITEM32SCORE    0.014672
18  CHECKITEM31SCORE    0.011661
15  CHECKITEM28SCORE    0.011230
14  CHECKITEM27SCORE    0.010625
13              PCO2    0.009641
10              HCO3    0.009167
25    INFECTIONSITE9    0.003572
23    INFECTIONSITE4    0.001200


In [47]:
importance_all = []

for i, drug in enumerate(y_train.columns):
    
    model = multi_model.estimators_[i]
    
    imp = pd.Series(
        model.feature_importances_,
        index=X_train.columns
    )
    
    imp_df = imp.reset_index()
    imp_df.columns = ['feature','importance']
    imp_df['drug'] = drug
    
    importance_all.append(imp_df)

importance_all = pd.concat(importance_all)

In [48]:
importance_summary = importance_all.groupby('feature')['importance'].mean().sort_values(ascending=False)

print(importance_summary)

feature
VITALSIGNSBT        0.113846
INFECTIONSITE2      0.106715
INFECTIONSITE1      0.091020
VITALSIGNSPR        0.087619
MAP                 0.071375
VITALSIGNSDBP       0.065394
WBC                 0.044415
CRP                 0.042597
INFECTIONSITE3      0.042567
VITALSIGNSSPO2      0.041366
Lymphocyte          0.039346
INFECTIONSITE5      0.036144
VITALSIGNSRR        0.029878
GPT                 0.028265
PT                  0.025093
VITALSIGNSGCS       0.023221
CHECKITEM29SCORE    0.016838
CHECKITEM32SCORE    0.015650
CHECKITEM27SCORE    0.014629
CHECKITEM30SCORE    0.013226
CHECKITEM28SCORE    0.012378
HCO3                0.008781
PCO2                0.008660
INFECTIONSITE4      0.008248
CHECKITEM31SCORE    0.006679
INFECTIONSITE9      0.006050
Name: importance, dtype: float64


In [49]:
# 看「群組 importance」

# 不要只看單一 feature。
# 而是算：
# infection site group importance

site_cols = [
'INFECTIONSITE1',
'INFECTIONSITE2',
'INFECTIONSITE3',
'INFECTIONSITE4',
'INFECTIONSITE5',
'INFECTIONSITE9'
]

importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
)

site_importance = importance[site_cols].sum()

print(site_importance)

0.5133441639046561


In [50]:
#只用感染部位建模型

site_cols = [
'INFECTIONSITE1',
'INFECTIONSITE2',
'INFECTIONSITE3',
'INFECTIONSITE4',
'INFECTIONSITE5',
'INFECTIONSITE9'
]

X_site = X_train[site_cols]
X_site_test = X_test[site_cols]

model = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=500, 
                           class_weight='balanced',
                           min_samples_leaf=5,
                           min_samples_split=10,
                           max_depth=15,
                           n_jobs=-1,
                           random_state=123
    )
)

model.fit(X_site, y_train)
y_pred = model.predict(X_site_test)

In [51]:
print(f1_score(y_test, y_pred, average='micro'))
print(f1_score(y_test, y_pred, average='macro'))

0.25008489303477616
0.24385489200884555


In [54]:
y_prob_list = model.predict_proba(X_site)

In [55]:
hamming_loss(y_test, y_pred)
jaccard_score(y_test, y_pred, average='samples', zero_division=0)
hit3 = hit_rate_at_k(y_test.values, y_prob_list, k=3)
print('Hit@3: ', hit3)
recall_3 = recall_at_k(y_test.values, y_prob_list, k=3)
print('Recall@3: ', recall_3)
map_3 = map_at_k(y_test.values, y_prob_list, k=3)
print('MAP@3: ', map_3)

Hit@3:  0.1637468716481945
Recall@3:  0.13189273464296852
MAP@3:  0.08383743893566156
